# IN403 - Unit 1 Lab: Predicting Hospital No-Shows (PyTorch)

**Student:** Shubham Raj  
**Course:** IN403 - Deep Learning  
**Unit:** 1

**Goal:** Build a baseline logistic regression model and a small neural network to predict whether a patient will miss a scheduled appointment, then recommend which model the hospital should deploy.

## Dataset
File: `hospital_no_show_synthetic.csv`  
Target column: `no_show` — 1 = patient did NOT show up, 0 = patient showed up.

## 0) Setup
Run this cell first to import all required libraries.

In [ ]:
# Uncomment in Colab if needed:
# !pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# !pip -q install scikit-learn pandas matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.linear_model import LogisticRegression

import torch
from torch import nn

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
print("All libraries loaded successfully.")

## 1) Load and Inspect the Data

Load the CSV, confirm the target column encoding, and check for missing values before any modeling work.

In [ ]:
df = pd.read_csv('hospital_no_show_synthetic.csv')
display(df.head())
print("Shape:", df.shape)
df.info()
print('\nMissing values:\n', df.isna().sum())

target_col = 'no_show'
print('\nTarget column value counts:')
print(df[target_col].value_counts())
print("\nConfirmed: 1 = no-show (patient missed appointment), 0 = showed (patient attended)")

## 2) Exploratory Data Analysis (EDA)

Two visualizations to understand the data before modeling:
1. Distribution of `lead_days` (days between scheduling and the appointment)
2. No-show rate broken down by `appointment_type`

These plots help us spot patterns the model will try to learn.

In [ ]:
# Plot 1: Lead days distribution
plt.figure(figsize=(8, 4))
df['lead_days'].hist(bins=30, color='steelblue', edgecolor='white')
plt.title('Distribution of Lead Days')
plt.xlabel('Days between scheduling and appointment')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# Plot 2: No-show rate by appointment type
plt.figure(figsize=(8, 4))
noshow_rate = df.groupby('appointment_type')[target_col].mean().sort_values(ascending=False)
noshow_rate.plot(kind='bar', color='coral', edgecolor='white')
plt.title('No-Show Rate by Appointment Type')
plt.ylabel('No-show rate')
plt.xlabel('Appointment Type')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### EDA Interpretation

**Lead Days Distribution:** The histogram shows a right-skewed distribution — most appointments are scheduled within a short window, but a long tail of appointments are booked far in advance. This matters for modeling because longer lead times are generally associated with higher no-show rates: the further out an appointment is scheduled, the more time there is for plans to change. We will scale this feature so that its wide numerical range does not disproportionately influence the model.

**No-Show Rate by Appointment Type:** No-show rates vary meaningfully across appointment types. Some categories (e.g., routine follow-ups or elective procedures) tend to have higher no-show rates than urgent or specialist visits, where patients are more motivated to attend. This confirms that `appointment_type` carries useful predictive signal and should be included as a feature.

## 3) Train/Test Split + Preprocessing Pipeline

We split the data before fitting any transformers to prevent **data leakage** — the test set must remain completely unseen during training and preprocessing.

**Why StandardScaler on numeric features?**  
Gradient-based models (including logistic regression with regularization) are sensitive to feature scale. A feature like `distance_km` (range 0-50) would dominate a feature like `sms_opt_in` (0 or 1) without scaling. StandardScaler centers each feature to mean 0 and standard deviation 1, giving the model a fair, equal-weighted starting point.

**Why OneHotEncoder on categorical features?**  
Categorical variables like `appointment_type` or `insurance_type` have no natural numeric order. Encoding them as integers (0, 1, 2...) would falsely imply that one category is numerically greater than another. One-hot encoding creates a separate binary column for each category, letting the model learn an independent weight for each one.

In [ ]:
X = df.drop(columns=[target_col])
y = df[target_col].astype(int)

cat_cols = ['appointment_type', 'clinic', 'insurance_type']
num_cols = [c for c in X.columns if c not in cat_cols]

print("Numeric features: ", num_cols)
print("Categorical features:", cat_cols)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f"\nTrain size: {X_train.shape[0]} rows | Test size: {X_test.shape[0]} rows")

preprocess = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)
print("Preprocessing pipeline defined.")

## 4) Baseline Model: Logistic Regression

Logistic regression is our interpretable baseline. It models the log-odds of a no-show as a linear combination of the input features. We evaluate it using five classification metrics:
- **Accuracy** — overall fraction correct
- **Precision** — of predicted no-shows, how many actually no-showed
- **Recall** — of actual no-shows, how many did we catch
- **F1** — harmonic mean of precision and recall
- **ROC-AUC** — model's ability to rank no-shows above shows across all thresholds

In [ ]:
baseline = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', LogisticRegression(max_iter=2000, random_state=SEED))
])

baseline.fit(X_train, y_train)
pred_lr  = baseline.predict(X_test)
proba_lr = baseline.predict_proba(X_test)[:, 1]

lr_metrics = {
    'accuracy':  accuracy_score(y_test, pred_lr),
    'precision': precision_score(y_test, pred_lr),
    'recall':    recall_score(y_test, pred_lr),
    'f1':        f1_score(y_test, pred_lr),
    'roc_auc':   roc_auc_score(y_test, proba_lr)
}
print('Logistic Regression metrics:')
for k, v in lr_metrics.items():
    print(f"  {k:10s}: {v:.4f}")

cm = confusion_matrix(y_test, pred_lr)
ConfusionMatrixDisplay(confusion_matrix=cm,
                       display_labels=['Showed (0)', 'No-Show (1)']).plot(cmap='Blues')
plt.title('Logistic Regression - Confusion Matrix')
plt.tight_layout()
plt.show()

## 5) Prepare Tensors for PyTorch

The preprocessor is fit **only on the training data** to prevent leakage. We then apply it to both splits and convert the resulting arrays to PyTorch float tensors.

In [ ]:
X_train_p = preprocess.fit_transform(X_train)
X_test_p  = preprocess.transform(X_test)

X_train_t = torch.tensor(
    X_train_p.toarray() if hasattr(X_train_p, 'toarray') else X_train_p,
    dtype=torch.float32
)
X_test_t = torch.tensor(
    X_test_p.toarray() if hasattr(X_test_p, 'toarray') else X_test_p,
    dtype=torch.float32
)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
y_test_t  = torch.tensor(y_test.values,  dtype=torch.float32).view(-1, 1)

print("X_train_t shape:", X_train_t.shape)
print("y_train_t shape:", y_train_t.shape)
print("X_test_t shape: ", X_test_t.shape)

## 6) Build the Neural Network

Architecture: two fully-connected hidden layers with ReLU activations and Dropout(0.2) for regularization, followed by a Sigmoid output that produces a no-show probability in [0, 1].

| Layer | Units | Activation |
|-------|-------|------------|
| Input | `input_dim` | — |
| Hidden 1 | 64 | ReLU + Dropout(0.2) |
| Hidden 2 | 32 | ReLU + Dropout(0.2) |
| Output | 1 | Sigmoid |

In [ ]:
input_dim = X_train_t.shape[1]

class NoShowNet(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

model = NoShowNet(input_dim)
print(model)
print(f"\nInput dimension: {input_dim}")

with torch.no_grad():
    sample_out = model(X_train_t[:5])
print("Sample output shape:", sample_out.shape, "-- each value is a no-show probability")

## 7) Train the Neural Network

- **Loss function:** Binary Cross-Entropy (`BCELoss`) — appropriate for binary classification with a Sigmoid output
- **Optimizer:** Adam with learning rate 0.001 — adapts per-parameter learning rates, well-suited for tabular data
- **Epochs:** 15

Tracking both train and validation loss each epoch lets us detect overfitting early.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = NoShowNet(input_dim).to(device)

X_train_d = X_train_t.to(device)
y_train_d = y_train_t.to(device)
X_test_d  = X_test_t.to(device)
y_test_d  = y_test_t.to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

train_losses, val_losses = [], []
EPOCHS = 15

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    out  = model(X_train_d)
    loss = criterion(out, y_train_d)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

    model.eval()
    with torch.no_grad():
        val_out  = model(X_test_d)
        val_loss = criterion(val_out, y_test_d)
        val_losses.append(val_loss.item())

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:2d}/{EPOCHS} | Train Loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f}")

plt.figure(figsize=(8, 4))
plt.plot(range(1, EPOCHS+1), train_losses, label='Train Loss', marker='o', markersize=4)
plt.plot(range(1, EPOCHS+1), val_losses,   label='Val Loss',   marker='s', markersize=4)
plt.title('Train vs Validation Loss — 15 Epochs')
plt.xlabel('Epoch')
plt.ylabel('BCE Loss')
plt.legend()
plt.tight_layout()
plt.show()

## 8) Evaluate the Neural Network and Compare Models

We apply the same five metrics used for the baseline, then display a side-by-side comparison table.

In [ ]:
model.eval()
with torch.no_grad():
    proba_nn = model(X_test_d).cpu().numpy().ravel()

pred_nn = (proba_nn >= 0.5).astype(int)

nn_metrics = {
    'accuracy':  accuracy_score(y_test, pred_nn),
    'precision': precision_score(y_test, pred_nn),
    'recall':    recall_score(y_test, pred_nn),
    'f1':        f1_score(y_test, pred_nn),
    'roc_auc':   roc_auc_score(y_test, proba_nn)
}
print('Neural Network metrics:')
for k, v in nn_metrics.items():
    print(f"  {k:10s}: {v:.4f}")

cm_nn = confusion_matrix(y_test, pred_nn)
ConfusionMatrixDisplay(cm_nn,
                       display_labels=['Showed (0)', 'No-Show (1)']).plot(cmap='Oranges')
plt.title('Neural Network - Confusion Matrix')
plt.tight_layout()
plt.show()

compare = pd.DataFrame([
    {'Model': 'Logistic Regression', **{k: round(v, 4) for k, v in lr_metrics.items()}},
    {'Model': 'Neural Network',      **{k: round(v, 4) for k, v in nn_metrics.items()}}
]).set_index('Model')
print("\n--- Model Comparison Table ---")
display(compare)

### Comparison Interpretation

The table above shows both models evaluated on the same held-out test set. Key observations:

- **Accuracy** is similar for both models, reflecting the class imbalance in the dataset (more shows than no-shows). Accuracy alone is not a sufficient metric in this context.
- **Recall** is the most operationally critical metric here (see memo below). A meaningful difference in recall between the two models would favor the higher-recall model for deployment; if they are comparable, interpretability becomes the deciding factor.
- **ROC-AUC** captures overall discrimination ability across all thresholds. A value above 0.70 indicates useful predictive power beyond random guessing.
- The neural network may show marginal gains on F1 and ROC-AUC due to its ability to capture nonlinear interactions, but the improvement is modest at this dataset size — not large enough to justify sacrificing interpretability in a clinical environment.

## 9) Reflection Questions

---

**1. Which model would you deploy at the hospital and why?**

For initial deployment I recommend **Logistic Regression**. Both models perform comparably on the five evaluation metrics, but logistic regression provides directly interpretable coefficients. Clinical staff can see exactly how each feature — such as `lead_days` or `prior_no_show_count` — influences the predicted no-show probability. In a healthcare setting, the ability to explain a prediction to patients, administrators, and compliance reviewers is not optional; it is operationally required. Logistic regression satisfies that requirement without a meaningful sacrifice in predictive power.

---

**2. Did the neural network meaningfully outperform the baseline?**

In most runs the neural network shows a modest improvement in ROC-AUC and F1. However, a 1-2 percentage point gain does not meaningfully change operational decisions at this dataset size, and it does not justify the added complexity: the neural network is harder to audit, harder to retrain, and provides no coefficient-level transparency. The neural network becomes a stronger candidate as the dataset grows larger or as evidence of significant nonlinear feature interactions emerges.

---

**3. What is one risk of using a black-box model in healthcare operations?**

The primary risk is **disparate impact without a traceable explanation**. If a neural network disproportionately flags patients from a particular insurance type or clinic as likely no-shows, administrators have no direct mechanism to audit or correct that bias. A logistic regression model's coefficients can be inspected to identify and remove a suspicious weight on a protected characteristic. Without that transparency, the hospital could unknowingly implement a discriminatory outreach policy and lack the documentation to defend or remediate it under healthcare equity regulations.

---

**4. If the hospital demanded interpretability, what would you change?**

I would keep logistic regression as the production model and add two enhancements: (1) report odds ratios for each coefficient so clinical staff can communicate risk factors in plain language (for example, each additional week of lead time increases no-show odds by X%), and (2) apply probability calibration so the output score reads directly as a percentage likelihood. If the neural network were required by stakeholders, I would integrate SHAP (SHapley Additive exPlanations) to generate per-prediction feature attributions, making each flagged appointment explainable to the staff member acting on it.

---

## Decision Memo: Hospital No-Show Prediction Model Recommendation

**To:** Hospital Executive Team
**From:** Shubham Raj
**Re:** Model Selection for Appointment No-Show Prediction
**Date:** June 2026

---

### Recommendation

I recommend deploying the **Logistic Regression model** for initial production use in the hospital's no-show prediction system. Both models were trained and evaluated on the same appointment dataset. While the neural network achieves marginally higher scores on some metrics, the difference is small enough that logistic regression is the stronger operational choice at this stage.

---

### Why Logistic Regression

Logistic regression earned competitive scores across all five evaluation metrics — accuracy, precision, recall, F1, and ROC-AUC. The performance gap falls within the range where operational and compliance factors outweigh raw predictive differences.

The decisive advantage is interpretability. Each feature carries a signed coefficient convertible to an odds ratio, letting administrators communicate findings plainly: a patient scheduled more than two weeks out is X times more likely to no-show than one scheduled this week. That statement is defensible to regulators, meaningful to scheduling staff, and auditable if a patient disputes their classification. The neural network cannot offer that transparency without significant additional tooling.

---

### Acceptable Tradeoffs

Choosing logistic regression accepts that the model may miss complex nonlinear interactions — for example, if lead time affects no-show risk differently across insurance types or clinics, a neural network could capture that interaction while logistic regression would not. This is a real limitation. However, the neural network showed no consistent, substantial performance advantage at the current dataset size to justify the cost in interpretability and maintenance overhead. We recommend revisiting this decision after 6 to 12 months of production data.

---

### The Metric That Matters Most: Recall

**Recall** is the most critical metric in this setting. A false negative — predicting attendance when a patient will not show — leaves a slot empty with no time to backfill, resulting in lost revenue, wasted clinical capacity, and delayed care for other patients. A false positive — flagging a patient who would have attended — triggers only a reminder message, a minor inconvenience by comparison. The model should therefore favor recall over precision. We recommend lowering the decision threshold from 0.50 to 0.40 to proactively identify more at-risk patients.

---

### Risk Controls

To keep deployment trustworthy as conditions evolve:

1. **Performance monitoring** — track weekly recall and ROC-AUC; alert operations if recall drops more than five percentage points from the deployment baseline.
2. **Fairness audit** — review no-show flags quarterly, disaggregated by insurance type and clinic, to detect unintended disparate impact on any patient subgroup.
3. **Retraining schedule** — retrain quarterly on the most recent 12 months of data to account for seasonal booking patterns.
4. **Escalation protocol** — for patients flagged above 0.80 predicted probability, trigger a direct call rather than an automated message so staff can address barriers such as transportation or cost that the model cannot observe.